# SIM V3 · Indoor · Phase B — FDTD dataset

Generate the full-wave (FDTD) training boxes for the U-Net surrogate.
Ground truth = complex field `U(x)` from `FullWaveScene`, phase-reduced to
`Ũ = U·e^{+jkd}`. Indoor: the whole 7th-floor plane per Tx (cheap fields).

Output: `fw_data_indoor/shard_*.npz` (`x[9,H,W]`, `y[2,H,W]`).

In [ ]:
# --- locate SIM V3 (works locally and on Colab) ---
# Colab: clone the repo, then set REPO_ROOT to it, e.g. '/content/Indoor_Walk_Test_7-7'.
REPO_ROOT = ''
import os, sys
if REPO_ROOT:
    SIMV3 = os.path.join(REPO_ROOT, 'Physics Engine', '2D', 'SIM V3')
else:
    SIMV3, d = os.path.abspath('..'), os.getcwd()
    for _ in range(6):
        if os.path.exists(os.path.join(d, '_bootstrap.py')): SIMV3 = d; break
        c = os.path.join(d, 'Physics Engine', '2D', 'SIM V3')
        if os.path.exists(os.path.join(c, '_bootstrap.py')): SIMV3 = c; break
        d = os.path.dirname(d)
assert os.path.exists(os.path.join(SIMV3, '_bootstrap.py')), f'set REPO_ROOT; not found: {SIMV3}'
sys.path.insert(0, SIMV3); os.chdir(SIMV3)
print('SIM V3 =', SIMV3)

In [ ]:
# Colab only: install deps (skip locally). torch usually preinstalled on Colab GPU.
# !pip -q install numpy scipy matplotlib tqdm onnxruntime
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

### Parameters — scale `n_tx` / `boxes_per_field` up for a real model.

In [ ]:
BANDS = ['LTE_B71_617']
SCENE = 'indoor'
N_TX = 6
BOXES_PER_FIELD = 80
BOX = 128
N_PER_WAVELENGTH = 8
REGION_M = 40   # outdoor only
OUT = 'fw_data_indoor'

### Generate (this runs FDTD — the expensive, one-time step)

In [ ]:
import fw_dataset
fw_dataset.generate(BANDS, scene=SCENE, n_tx=N_TX,
                    boxes_per_field=BOXES_PER_FIELD, box=BOX,
                    n_per_wavelength=N_PER_WAVELENGTH, region_m=REGION_M,
                    out_dir=OUT, seed=1)

### Inspect one training box (materials · |Ũ| target · log-distance)

In [ ]:
import glob, numpy as np, matplotlib.pyplot as plt
d = np.load(sorted(glob.glob(OUT + '/shard_*.npz'))[0]); X, Y = d['x'], d['y']
i = 0; fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].imshow(np.argmax(X[i, :6], 0).T, origin='lower'); ax[0].set_title('materials')
ax[1].imshow(np.hypot(Y[i, 0], Y[i, 1]).T, origin='lower'); ax[1].set_title('|U~| (target)')
ax[2].imshow(X[i, 8].T, origin='lower'); ax[2].set_title('log-distance ch'); plt.show()
print('tensors:', X.shape, Y.shape)